In [ ]:
#埋め込みの読み込み
import numpy as np
from gensim.models import KeyedVectors

def load_pretrained_embeddings(embedding_path, vocab_limit=None):
    embeddings = []
    token2id = {}
    id2token = {}

    # GoogleNews-vectors-negative300.bin を読み込む
    word_vectors = KeyedVectors.load_word2vec_format(embedding_path, binary=True)

    embedding_dim = word_vectors.vector_size
    embeddings.append(np.zeros(embedding_dim))  # <PAD>トークン用ゼロベクトル
    token2id['<PAD>'] = 0
    id2token[0] = '<PAD>'

    for idx, word in enumerate(word_vectors.index_to_key):
        if vocab_limit and (idx >= vocab_limit):
            break
        vector = word_vectors[word]

        token_id = len(embeddings)
        token2id[word] = token_id
        id2token[token_id] = word
        embeddings.append(vector)

    embedding_matrix = np.vstack(embeddings) #各単語の埋め込みを縦方向に結合(ベクトル→行列)
    return embedding_matrix, token2id, id2token

# 使い方
embedding_path = 'GoogleNews-vectors-negative300.bin'
embedding_matrix, token2id, id2token = load_pretrained_embeddings(embedding_path, vocab_limit=50000)


In [ ]:
#データセットの読み込み
import csv
import torch

def load_sst_data(file_path, token2id):
    dataset = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            text = row['sentence']
            label = int(row['label'])

            # 単語ごとにトークンIDに変換
            tokens = text.split()
            input_ids = [token2id[token] for token in tokens if token in token2id]

            # 全部消えたらスキップ
            if len(input_ids) == 0:
                continue

            item = {
                'text': text,
                'label': torch.tensor([float(label)]),
                'input_ids': torch.tensor(input_ids)
            }
            dataset.append(item)
    return dataset

# 使い方
train_file = './SST-2/train.tsv'
dev_file = './SST-2/dev.tsv'

train_data = load_sst_data(train_file, token2id)
dev_data = load_sst_data(dev_file, token2id)


In [8]:
# trainデータの確認
for item in train_data[:10]:
    print(f"Text: {item['text']}, Label: {item['label'].item()}, Input IDs: {item['input_ids']}")

Text: hide new secretions from the parental units , Label: 0.0, Input IDs: tensor([ 5785,    66,    18,    12, 15095,  1594])
Text: contains no wit , only labored gags , Label: 0.0, Input IDs: tensor([ 3475,    87, 15888,    90, 27695, 42637])
Text: that loves its characters and communicates something rather beautiful about human nature , Label: 1.0, Input IDs: tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,
         1964])
Text: remains utterly satisfied to remain the same throughout , Label: 0.0, Input IDs: tensor([  987, 14528,  4941,   873,    12,   208,   898])
Text: on the worst revenge-of-the-nerds clichés the filmmakers could dredge up , Label: 0.0, Input IDs: tensor([    6,    12,  1445, 43789,    12, 10946,    76, 41349,    42])
Text: that 's far too tragic to merit such superficial treatment , Label: 0.0, Input IDs: tensor([    4,   354,   254,  6521,  8441,   137, 24459,  1015])
Text: demonstrates that the director of such hollywood blockbuster

In [10]:
# devデータの確認
for item in dev_data[:10]:
    print(f"Text: {item['text']}, Label: {item['label'].item()}, Input IDs: {item['input_ids']}")

Text: it 's a charming and often affecting journey . , Label: 1.0, Input IDs: tensor([   16, 13259,   640,  5199,  3900])
Text: unflinchingly bleak and desperate , Label: 0.0, Input IDs: tensor([12607,  4984])
Text: allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker . , Label: 1.0, Input IDs: tensor([ 1488,   165,   684,     4,     5,  6091, 14671,   339,   513,    15,
         1073,   507, 24346, 11212])
Text: the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales . , Label: 1.0, Input IDs: tensor([   12,  2527, 10358,   637, 37102,  1868,    20,    53, 18600,   483,
           12,   621, 37066, 30723])
Text: it 's slow -- very , very slow . , Label: 0.0, Input IDs: tensor([  16, 1804,  139,  139, 1804])
Text: although laced with humor and a few fanciful touches , the film is a refreshingly serious look at young women . , Label: 1.0, Input IDs: tensor([ 1022, 17909,     9,